In [1]:
import joblib
import sys
sys.path.append("../")

from src.data_prep import load_dataset, sample_dataset, train_test_split
from src.embeddings import load_embedding_model, encode_texts
from src.classifier import train_classifier
from src.pipeline import classify_ticket
from src.pipeline import predict_ticket
from src.metrics import evaluate_model

c:\Users\Alexa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_dataset("../data/tickets.csv")

df = sample_dataset(df)

train_df, test_df = train_test_split(df)

Topic_group
Access                   25
Administrative rights    25
HR Support               25
Hardware                 25
Internal Project         25
Miscellaneous            25
Purchase                 25
Storage                  25
Name: count, dtype: int64
Train size: 140
Test size: 60


In [3]:
test_df.iloc[3]["Topic_group"]

'Purchase'

In [4]:
test_df.iloc[3]["Document"]

'new purchase po thursday pm purchase po dear purchased hub link please log installation please take consideration mandatory receipts section order receive item ordered how video kind regards administrator'

In [5]:
model = load_embedding_model()

X_train = encode_texts(model, train_df["Document"].tolist())

y_train = train_df["Topic_group"]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11441.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
clf = train_classifier(X_train, y_train)

In [7]:
joblib.dump(clf, "../models/classifier.joblib")

['../models/classifier.joblib']

In [8]:
X_test = encode_texts(model, test_df["Document"].tolist())
y_test = test_df["Topic_group"].tolist()

y_pred = []

for emb in X_test:
    pred = predict_ticket(clf, emb.reshape(1, -1))
    y_pred.append(pred)

# calcular métricas
report = evaluate_model(y_test, y_pred)

print(report)


                       precision    recall  f1-score   support

               Access       0.67      0.86      0.75         7
Administrative rights       0.20      0.50      0.29         4
           HR Support       0.50      0.56      0.53         9
             Hardware       0.60      0.27      0.38        11
     Internal Project       0.80      0.67      0.73         6
        Miscellaneous       0.62      0.56      0.59         9
             Purchase       0.88      0.78      0.82         9
              Storage       1.00      1.00      1.00         5

             accuracy                           0.62        60
            macro avg       0.66      0.65      0.63        60
         weighted avg       0.66      0.62      0.62        60



In [9]:
ticket = test_df.iloc[3]["Document"]

classify_ticket(ticket, model, clf)

{'class': 'Purchase',
 'justification': 'This ticket belongs to the "Purchase" class because it involves a new purchase made on Thursday PM, and the administrator is requesting assistance with logging the installation and receiving the item ordered. \n\nEvidence from the text includes: \n- "new purchase" \n- "purchase PO" \n- "item ordered" \n- "receive item"'}

In [10]:
for ticket in test_df['Document'][:10]:
    print(test_df['Topic_group'][:10])

95                  Hardware
15                    Access
30     Administrative rights
158                 Purchase
128            Miscellaneous
115         Internal Project
69                HR Support
170                 Purchase
174                 Purchase
45     Administrative rights
Name: Topic_group, dtype: object
95                  Hardware
15                    Access
30     Administrative rights
158                 Purchase
128            Miscellaneous
115         Internal Project
69                HR Support
170                 Purchase
174                 Purchase
45     Administrative rights
Name: Topic_group, dtype: object
95                  Hardware
15                    Access
30     Administrative rights
158                 Purchase
128            Miscellaneous
115         Internal Project
69                HR Support
170                 Purchase
174                 Purchase
45     Administrative rights
Name: Topic_group, dtype: object
95                  Hardware
15

In [ ]:
results = []

for ticket in test_df["Document"][:10]:
    results.append(classify_ticket(ticket, model, clf))

In [ ]:
results